# 04 · District cleaning-interval map
For each of India's 820 districts, simulate one year (2024) day by day:
- dust builds up at the district's soiling rate: **rate (%/day) = A + B × dust deposition** (fitted in notebook 02)
- rain ≥ 5 mm/day washes the panel clean
- manual cleaning every **T** days

Choose the T (1–90 days, or never) that minimises **cleaning cost + value of lost electricity**.

Inputs: `data/district_daily_2024.csv` (NASA POWER rain + irradiance at each district centre) and `data/merra2_district_monthly_dust_2024.csv` (notebook 02).
Outputs: `data/district_cleaning_intervals.csv`, `figures/cleaning_map.html`, `data/sensitivity.csv`.

In [ ]:
!pip -q install geopandas folium
import geopandas as gpd, pandas as pd, numpy as np, requests, time, concurrent.futures as cf
from scipy.stats import spearmanr
REPO = "https://raw.githubusercontent.com/ishjeetsingh/india-solar-soiling/main/"
gdf = gpd.read_file("https://raw.githubusercontent.com/datta07/INDIAN-SHAPEFILES/master/INDIA/INDIA_DISTRICTS.geojson").to_crs(4326)
cent = gdf.to_crs(3857).centroid.to_crs(4326)
gdf["clat"], gdf["clon"] = cent.y.round(3), cent.x.round(3)
print(len(gdf), "districts")

## Daily rain + irradiance per district
Loaded from the repo. Set `REBUILD = True` to re-download from NASA POWER (~10 min; failed requests are retried).

In [ ]:
REBUILD = False

def get_power(lat, lon, start="20240101", end="20241231"):
    url = "https://power.larc.nasa.gov/api/temporal/daily/point"
    params = {"parameters": "PRECTOTCORR,ALLSKY_SFC_SW_DWN", "community": "RE",
              "latitude": lat, "longitude": lon, "start": start, "end": end, "format": "JSON"}
    r = requests.get(url, params=params, timeout=120)
    r.raise_for_status()
    df = pd.DataFrame(r.json()["properties"]["parameter"])
    df.index = pd.to_datetime(df.index)
    return df.replace(-999, pd.NA).rename(columns={"PRECTOTCORR": "rain_mm", "ALLSKY_SFC_SW_DWN": "irradiance_kwh_m2"})

def fetch(i):
    for attempt in range(3):
        try:
            df = get_power(gdf.clat[i], gdf.clon[i])
            df["district_id"] = i
            return df.rename_axis("date").reset_index()
        except Exception:
            time.sleep(3)
    return None

if REBUILD:
    with cf.ThreadPoolExecutor(5) as ex:
        results = list(ex.map(fetch, range(len(gdf))))
    missing = [i for i, r in enumerate(results) if r is None]
    for i in missing:                                   # retry slowly
        results[i] = fetch(i); time.sleep(1)
    dist_daily = pd.concat([r for r in results if r is not None], ignore_index=True)
    dist_daily.to_csv("district_daily_2024.csv", index=False)
else:
    dist_daily = pd.read_csv(REPO + "data/district_daily_2024.csv", parse_dates=["date"])
print(dist_daily.district_id.nunique(), "districts with weather data")

In [ ]:
dust = pd.read_csv(REPO + "data/merra2_district_monthly_dust_2024.csv").set_index("district_id")
months = [c for c in dust.columns if c.startswith("2024-")]

## Assumptions
| Parameter | Base value | Note |
|---|---|---|
| A, B | 0.301, 0.0075 | fitted in notebook 02 (exploratory, 6 locations) |
| Performance ratio | 0.80 | typical utility PV |
| Tariff | ₹2.6/kWh | placeholder; tested 2.0–3.5 |
| Cleaning cost | ₹2 per kWp per clean | placeholder (~₹1–1.2 per panel reported); tested 1–4 |
| Rain threshold | 5 mm/day | tested 2–10 |

In [ ]:
A, B = 0.301, 0.0075
PR, TARIFF, CLEAN, RAIN_THR = 0.80, 2.6, 2.0, 5.0

def optimise(rain, irr, rate, Ts=list(range(1, 91)) + [999]):
    best = None
    for T in Ts:
        L = lost = 0.0; n = since = 0
        for r, g, k in zip(rain, irr, rate):
            since += 1
            if since >= T: L, since, n = 0.0, 0, n + 1     # manual clean
            L += k                                          # dust builds up (%)
            if r >= RAIN_THR: L = 0.0                       # rain washes panel
            lost += g * PR * L / 100                        # kWh lost per kWp
        cost = n * CLEAN + lost * TARIFF
        if best is None or cost < best[0]:
            best = (cost, T, n, lost)
    return best

def district_inputs(i, g, A_=None, B_=None):
    A_ = A if A_ is None else A_; B_ = B if B_ is None else B_
    rate = A_ + B_ * dust.loc[i, months].astype(float).values[g.date.dt.month.values - 1]
    return g.rain_mm.astype(float).fillna(0).values, g.irradiance_kwh_m2.astype(float).fillna(0).values, rate

out = []
for i, g in dist_daily.groupby("district_id"):
    g = g.sort_values("date")
    rain, irr, rate = district_inputs(i, g)
    cost, T, n, lost = optimise(rain, irr, rate)
    out.append((i, T, n, lost / (irr.sum() * PR) * 100, cost))

res = pd.DataFrame(out, columns=["district_id", "best_interval_days", "cleanings_per_year",
                                 "energy_lost_pct", "annual_cost_inr_per_kWp"])
gdf2 = gdf.join(res.set_index("district_id"))
gdf2.drop(columns="geometry").round(3).to_csv("district_cleaning_intervals.csv", index=False)
print(res.best_interval_days.replace(999, np.nan).describe().round(1))
gdf2.nsmallest(5, "best_interval_days")[["district", "state", "best_interval_days"]]

## Map

In [ ]:
bands = [0, 7, 14, 21, 30, 60, 1000]
names = ["≤1 week", "1–2 weeks", "2–3 weeks", "3–4 weeks", "1–2 months", "2+ months"]
gdf2["clean_every"] = pd.cut(gdf2.best_interval_days, bands, labels=names).astype(str)
gdf2["label"] = gdf2.best_interval_days.map(lambda t: "rain is enough" if t == 999 else f"{t:.0f} days")
gdf2["geometry"] = gdf2.simplify(0.01)

m = gdf2.explore(column="clean_every", categorical=True, categories=names, cmap="RdYlGn",
                 tiles=None, legend=True, legend_kwds={"caption": "Optimal cleaning interval"},
                 tooltip=["district", "state", "label", "energy_lost_pct"])
m.save("cleaning_map.html")
m

## Sensitivity analysis (200 random districts)
**Median interval** = how much the typical answer shifts. **Rank agreement** (Spearman) = whether the *ordering* of districts stays the same.

In [ ]:
rng = np.random.default_rng(0)
ids = rng.choice(dist_daily.district_id.unique(), 200, replace=False)
groups = {i: g.sort_values("date") for i, g in dist_daily[dist_daily.district_id.isin(ids)].groupby("district_id")}

def run(A_=None, B_=None, clean=2.0, tariff=2.6, thr=5.0):
    global CLEAN, TARIFF, RAIN_THR
    CLEAN, TARIFF, RAIN_THR = clean, tariff, thr
    return pd.Series([optimise(*district_inputs(i, g, A_, B_))[1] for i, g in groups.items()], index=list(groups))

base = run()
scen = {"cleaning ₹1/kWp": dict(clean=1.0), "cleaning ₹4/kWp": dict(clean=4.0),
        "tariff ₹2.0/kWh": dict(tariff=2.0), "tariff ₹3.5/kWh": dict(tariff=3.5),
        "rain threshold 2 mm": dict(thr=2.0), "rain threshold 10 mm": dict(thr=10.0),
        "no dust term (B=0)": dict(B_=0.0)}
rows = [("base case", base.median(), 1.0)]
for name, kw in scen.items():
    s = run(**kw)
    rows.append((name, s.median(), spearmanr(base, s).correlation))
CLEAN, TARIFF, RAIN_THR = 2.0, 2.6, 5.0
sens = pd.DataFrame(rows, columns=["scenario", "median_interval_days", "rank_agreement"]).round(2)
sens.to_csv("sensitivity.csv", index=False)
sens